# LLM overlay pilot — data in pandas

The production-overlay replication (phase 2, 2026-07-16): each stock's PASS/WATCHLIST/REVIEW/AVOID_RED_FLAG rating and the other categorical judgments, merged with book membership and forward returns. Run all cells top-to-bottom from the repo root; the final DataFrame is also written to `output/llm_pilot/llm_pilot_master.csv`.

**DataFrames**

| DataFrame | one row per | source |
|---|---|---|
| `cohorts` | (cohort, ticker) book membership | `output/llm_pilot/cohorts.csv` |
| `overlay` | production-overlay judgments | `output/llm_pilot/overlay/*.json` |
| `master` | both merged + demeaned returns | built below |

**Key columns**
- `cohort` — "2024" (formation 2024-06-28, fwd window 2024-07→2025-06) or "2025" (formation 2025-06-30, fwd 2025-07→2026-06)
- `composite_pct` — quant composite percentile *within the book* at formation (PIT, clean caches)
- `fwd_12m_ret` — 12-month forward total return from formation
- `ret_dm` — `fwd_12m_ret` demeaned within cohort (use this for pooled stats; the two cohorts have very different base years)
- `final_research_status` — PASS / WATCHLIST / REVIEW / AVOID_RED_FLAG
- `quant_signal_review` — CONFIRMS / MIXED / WEAKENS / CONTRADICTS (does the qualitative evidence support the quant signal?)
- `thesis_alignment` (BULLISH/NEUTRAL/BEARISH), `qualitative_risk_level`, `business_quality`, `management_tone`, `accounting_risk_cat`, `filing_risk` (all LOW/MEDIUM/HIGH-style categoricals)
- `overlay_confidence` — 0-100; `overlay_n_red_flags` / `overlay_red_flags` — the flag list

**Caution:** 54 tickers appear in both cohorts (forward windows don't overlap in time, but they are not independent observations). The two pre-registered reads both FAILED (see `docs/llm_pilot_design.md`); everything else here is exploratory.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()          # run from the repo root
sys.path.insert(0, str(ROOT))
OUT = ROOT / "output" / "llm_pilot"
pd.set_option("display.width", 160)

In [ ]:
# --- cohort books (PIT, post ghost-member fix) ---
cohorts = pd.read_csv(OUT / "cohorts.csv", dtype={"cohort": str})
print(cohorts.shape)
cohorts.head()

In [ ]:
# --- phase 2: production-overlay categoricals ---
rows = []
for p in sorted((OUT / "overlay").glob("*.json")):
    cohort, ticker = p.stem.split("_", 1)
    d = json.load(open(p))
    if "error" in d:
        continue
    rows.append({
        "cohort": cohort, "ticker": ticker,
        "final_research_status": d.get("final_research_status"),
        "quant_signal_review": d.get("quant_signal_review"),
        "thesis_alignment": d.get("thesis_alignment"),
        "qualitative_risk_level": d.get("qualitative_risk_level"),
        "business_quality": d.get("business_quality"),
        "management_tone": d.get("management_tone"),
        "accounting_risk_cat": d.get("accounting_risk"),
        "filing_risk": d.get("filing_risk"),
        "insider_signal": d.get("insider_signal_interpretation"),
        "overlay_confidence": d.get("confidence"),
        "overlay_n_red_flags": len(d.get("red_flags") or []),
        "overlay_red_flags": " | ".join(d.get("red_flags") or []),
    })
overlay = pd.DataFrame(rows)
print(overlay.shape)
overlay.head()

In [ ]:
# --- master: cohorts + overlay merged, demeaned return added, saved to CSV ---
master = cohorts.merge(overlay, on=["cohort", "ticker"], how="left")
master["ret_dm"] = (master.fwd_12m_ret
                    - master.groupby("cohort").fwd_12m_ret.transform("mean"))
master.to_csv(OUT / "llm_pilot_master.csv", index=False)
print(master.shape, "->", OUT / "llm_pilot_master.csv")
master.head()

In [ ]:
from scripts.llm_pilot_charts import monthly_prices, sharpe

WINDOWS = {"2024": ("2024-06-30", "2025-06-30"),
           "2025": ("2025-06-30", "2026-06-30")}

mpx = monthly_prices(sorted(master.ticker.unique()))

def stitched_series(sub: pd.DataFrame) -> pd.Series:
    """24-month EW monthly-return series for the names in `sub`."""
    parts = []
    for c, (start, end) in WINDOWS.items():
        names = [t for t in sub.loc[sub.cohort == c, "ticker"]
                 if t in mpx.columns]
        if names:
            parts.append(mpx.loc[start:end, names].pct_change().iloc[1:].mean(axis=1))
    return pd.concat(parts)

book = stitched_series(master)
print(f"whole-book EW Sharpe: {sharpe(book):.2f}")

## Example analyses (delete and replace with your own)

In [ ]:
# demeaned forward return + Sharpe by research status
for status, grp in master.dropna(subset=["final_research_status"]).groupby("final_research_status"):
    s = stitched_series(grp)
    sr = sharpe(s) if len(s) > 12 else float("nan")
    print(f"{status:<15s} n={len(grp):3d}  ret_dm={grp.ret_dm.mean():+.3f}  SR={sr:.2f}")

In [ ]:
# mean demeaned forward return by overlay category (pick any categorical column)
for field in ["final_research_status", "quant_signal_review",
              "management_tone", "accounting_risk_cat"]:
    t = (master.dropna(subset=[field, "ret_dm"])
         .groupby(field).agg(n=("ret_dm", "count"), mean_ret_dm=("ret_dm", "mean"))
         .round(3))
    print(f"\n{field}:\n{t.to_string()}")

In [ ]:
# read one stock's full overlay output (all fields incl. evidence bullets)
TICKER, COHORT = "ACGL", "2024"
print(json.dumps(json.load(open(OUT / "overlay" / f"{COHORT}_{TICKER}.json")), indent=1))